# GPU Black Hole Time-Dependent Simulation
GPU-accelerated version of the BH base (V404 Cygni-like) accretion disk simulation.
Uses `blackhole.gpu.simulation.run_simulation` which keeps all arrays GPU-resident
(or runs on CPU via NumPy when CuPy is unavailable).

In [1]:
import time

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

from blackhole.constants import M_sun
from blackhole.gpu import is_gpu_available
from blackhole.gpu.simulation import SimulationConfig, run_simulation

print(f"GPU available: {is_gpu_available()}")
if is_gpu_available():
    import cupy as cp
    print(f"CuPy device: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")

GPU available: True
CuPy device: NVIDIA GeForce RTX 5080 Laptop GPU


In [ ]:
cfg = SimulationConfig(
    M_star=9 * M_sun,
    R_1=5e8,
    R_K=2.2e11,
    R_N=4.2e11,
    M_dot=3e17,
    alpha_cold=0.04,
    alpha_hot=0.2,
    N_base=10_000,
    N_n=3,
    min_Sigma=1e-5,
    timesteps=1_001,
    output_interval=5,
    tidal_params={"cw": 0.2, "a_1": 2.2e12, "n_1": 5, "trunc_frac": 9.1 / 10},
    enable_irradiation=False,
    enable_evaporation=False,
    theta=0.5,
    use_gpu=True,
)

t0 = time.time()
result = run_simulation(cfg)
elapsed = time.time() - t0
print(f"Simulation complete: {len(result.t_history)} snapshots in {elapsed:.1f} s")
print(f"Total simulated time: {result.t_history[-1]:.2e} s")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

r_grid = np.linspace(cfg.R_1, cfg.R_N, len(result.Sigma_history[0]))

# Surface density evolution
ax = axes[0, 0]
for i in range(0, len(result.Sigma_history), max(1, len(result.Sigma_history) // 10)):
    ax.semilogy(r_grid, result.Sigma_history[i], alpha=0.5)
ax.set_xlabel("r (cm)")
ax.set_ylabel("Sigma (g/cm^2)")
ax.set_title("Surface Density Evolution")

# Temperature evolution
ax = axes[0, 1]
for i in range(0, len(result.Temp_history), max(1, len(result.Temp_history) // 10)):
    ax.semilogy(r_grid, result.Temp_history[i], alpha=0.5)
ax.set_xlabel("r (cm)")
ax.set_ylabel("T_c (K)")
ax.set_title("Midplane Temperature Evolution")

# Inner-edge Sigma vs time
ax = axes[1, 0]
ax.plot(result.t_history, result.Sigma_transfer_history)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Sigma_inner (g/cm^2)")
ax.set_title("Inner-Edge Surface Density")

# Scale height evolution
ax = axes[1, 1]
for i in range(0, len(result.H_history), max(1, len(result.H_history) // 10)):
    ax.semilogy(r_grid, result.H_history[i], alpha=0.5)
ax.set_xlabel("r (cm)")
ax.set_ylabel("H (cm)")
ax.set_title("Scale Height Evolution")

plt.tight_layout()
plt.show()

In [ ]:
import os

DATA_DIR = "../../data"
os.makedirs(DATA_DIR, exist_ok=True)
SUFFIX = "_history_bath_array_bh_gpu.csv"

np.savetxt(f"{DATA_DIR}/Sigma{SUFFIX}", np.array(result.Sigma_history), delimiter=",")
np.savetxt(f"{DATA_DIR}/Temp{SUFFIX}", np.array(result.Temp_history), delimiter=",")
np.savetxt(f"{DATA_DIR}/H{SUFFIX}", np.array(result.H_history), delimiter=",")
np.savetxt(f"{DATA_DIR}/alpha{SUFFIX}", np.array(result.alpha_history), delimiter=",")
np.savetxt(f"{DATA_DIR}/t{SUFFIX}", np.array(result.t_history), delimiter=",")
np.savetxt(f"{DATA_DIR}/Sigma_transfer{SUFFIX}", np.array(result.Sigma_transfer_history), delimiter=",")

print(f"Saved 6 CSV files to {DATA_DIR}/ with suffix '{SUFFIX}'")

## Notes
- This notebook uses `blackhole.gpu.simulation.run_simulation` which orchestrates the full
  timestep loop with GPU-resident arrays (when CuPy is available).
- **Auto-dt**: the timestep ceiling is automatically computed from physics constraints
  (mass deposition limit and thermal timescale at R_K). No manual dt tuning needed —
  changing N_base, M_dot, or geometry auto-adapts the timestep.
- The `add_mass` step transfers ~824 bytes to CPU each timestep (inherently sequential).
- All other physics (diffusion, solvers, viscosity) runs on device.
- Set `use_gpu=False` to run the same orchestrator with NumPy arrays on CPU.